This section will cover:

 - What dimensionality reduction is and why we use it
 - Principal Component Analysis (PCA) as a technique to reduce complexity
 - Clustering methods to group similar observations
 - K-Means clustering for partitioning data
 - Expectation-Maximization (EM) for probabilistic clustering
 - DBSCAN for density-based clustering
 - Visualizing and interpreting results

In the previous sections, we've learned how to work with and analyze data. Now, we'll tackle a more sophisticated challenge: **what do we do when we have lots of variables?** Imagine a dataset with 50 or 100 columns describing neighborhoods, products, or observations. It's hard to visualize, hard to understand, and computationally expensive to work with. This section introduces two complementary techniques: **dimensionality reduction** to simplify complex data, and **clustering** to find natural groupings within that data.

## Why dimensionality reduction and clustering?

Consider this scenario: you have data on neighborhoods with dozens of variables - demographics, housing, transit, crime, employment, amenities, and more. These variables are often **interdependent**. Income might correlate with education, transit access might relate to housing density, and so on. When variables are interconnected like this, you often don't need all of them.

**Dimensionality reduction** addresses this by finding the underlying patterns that explain most of the variation in your data. Instead of 50 columns, you might reduce to 2 or 3 that capture most of the essential structure.

**Clustering** then helps you discover natural groupings in this simplified data. Are there distinct neighborhood types? Do neighborhoods cluster into urban, suburban, and rural patterns? Clustering finds these patterns *without* being told in advance what categories to look for.

Together, these techniques allow you to:
 - Visualize high-dimensional data on a 2D plot
 - Discover hidden patterns and structure
 - Simplify complex datasets for analysis
 - Segment data into meaningful groups for planning, targeting, or understanding

## The workflow: From raw data to insights

Here's the typical workflow we'll use:

1. **Load and prepare** your data (ensure all variables are numeric and standardized)
2. **Apply PCA** to reduce dimensions while preserving variance
3. **Decide how many dimensions** you need (often 2-3 for visualization)
4. **Apply a clustering algorithm** (K-Means, EM, or DBSCAN) to group observations
5. **Visualize and interpret** the clusters in reduced space
6. **Iterate**: adjust parameters and experiment with different datasets

Let's get started!

## Visual Intuition: What Dimensionality Reduction and Clustering Look Like

Before we dive into code, let's build intuition with some visual examples.

**Principal Component Analysis (PCA)** transforms high-dimensional data into lower dimensions while preserving the most important variation:

![PCA Scaling Example](../advanced-statistics-multivariate-methods/img/PCA-scaling.png)

The left image shows raw data without proper scaling - variables with large ranges dominate. The right image shows standardized data where PCA can properly identify the main patterns. Notice how the data clusters naturally along two main directions.

**K-Means Clustering** partitions data into k groups by finding centroids:

![K-Means Example](../advanced-statistics-multivariate-methods/img/k-means.png)

The algorithm repeatedly assigns points to the nearest center, then updates centers. The result is compact, roughly spherical clusters.

**Expectation-Maximization (EM)** takes a softer approach, estimating probability of belonging to each cluster rather than making hard assignments:

![EM Example](../advanced-statistics-multivariate-methods/img/em.png)

This is useful when clusters overlap or boundaries are fuzzy - you get probabilities rather than strict assignments.

## Setup: Installing and importing libraries

First, we need to install `scikit-learn`, which provides implementations of PCA and clustering algorithms. If you've already installed it, you can skip this step.

In [ ]:
# !pip install scikit-learn matplotlib seaborn

Now let's import the libraries we'll use throughout this notebook:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## Configure dataset and column names

To make it easy to swap datasets and automatically detect column names, we'll set up configuration variables at the start. Once you change the dataset, these variables will be used throughout the notebook:

In [ ]:
# CHOOSE YOUR DATASET HERE
data_file = "data/iris.csv"
# Try: "data/iris.csv", "data/penguins.csv", "data/tips.csv", "data/titanic.csv", "data/diamonds.csv"

# Load the dataset
df = pd.read_csv(data_file)

# Auto-detect category/label column (for coloring plots)
# Common names in datasets
label_column = None
for col_name in ['cut', 'species', 'day', 'class']:
    if col_name in df.columns:
        label_column = col_name
        break

print(f"Dataset: {data_file}")
print(f"Shape: {df.shape}")
print(f"Label column detected: {label_column}")
print(f"\nFirst few rows:")
df.head()

# Part 1: Dimensionality Reduction with PCA

## What is Principal Component Analysis (PCA)?

**Principal Component Analysis** is a dimensionality reduction technique that transforms your data into a new coordinate system where the axes (called "principal components") are ordered by how much variation they explain.

Think of it this way: If you have data points scattered in 3D space and they mostly line up along a plane, you don't really need all 3 dimensions - you can capture almost all the information in 2D.

**Key ideas:**
 - The **first principal component** captures the direction of maximum variance
 - The **second principal component** captures the second-most variance (perpendicular to the first)
 - And so on...
 - By keeping the top 2 or 3 components, you retain most information while making data visualizable

**Important:** PCA requires standardization - all variables must be on the same scale, otherwise variables with larger ranges will dominate.

## Loading and exploring the iris dataset

We'll use the **Iris dataset**, a classic dataset with measurements of iris flowers. It has 4 numeric variables (sepal length, sepal width, petal length, petal width) across 3 species.

**Try changing `data_file` above to one of the other datasets (diamonds, penguins, tips, or titanic) to explore how PCA works on different data!**

## Preparing data for PCA

For PCA to work properly, we need:
1. Only **numeric columns** (no text or categories)
2. **No missing values**
3. **Standardized data** (all on the same scale)

For the iris dataset, all columns are already numeric. Let's prepare the data:

**For other datasets:** You may need to drop non-numeric columns or handle missing values. The code below shows how.

In [ ]:
# Select only numeric columns
df_numeric = df.select_dtypes(include=[np.number])

# Remove rows with missing values
df_numeric = df_numeric.dropna()

print(f"Numeric data shape: {df_numeric.shape}")
print(f"\nData statistics:")
print(df_numeric.describe())

Now let's **standardize** the data. This is crucial - without it, variables with large ranges will dominate.

In [ ]:
# Standardize the data (mean=0, std=1)
scaler = StandardScaler()
data_scaled = scaler.fit_transform(df_numeric)

print(f"Scaled data mean: {data_scaled.mean(axis=0).round(3)}")
print(f"Scaled data std: {data_scaled.std(axis=0).round(3)}")

## Applying PCA

Now we'll apply PCA. We'll keep all components first to see how much variance each explains, then we'll reduce to 2D for visualization.

In [ ]:
# Apply PCA, keeping all components
pca_full = PCA()
pca_full.fit(data_scaled)

# Print explained variance
print("Variance explained by each component:")
for i, var in enumerate(pca_full.explained_variance_ratio_):
    cumsum = pca_full.explained_variance_ratio_[:i+1].sum()
    print(f"  PC{i+1}: {var:.4f} (cumulative: {cumsum:.4f})")

Let's visualize this explained variance:

In [ ]:
# Plot explained variance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Scree plot
ax1.plot(range(1, len(pca_full.explained_variance_ratio_) + 1), 
         pca_full.explained_variance_ratio_, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Explained Variance Ratio')
ax1.set_title('Scree Plot: Variance by Component')
ax1.grid(alpha=0.3)

# Cumulative variance
cumsum = np.cumsum(pca_full.explained_variance_ratio_)
ax2.plot(range(1, len(cumsum) + 1), cumsum, 'go-', linewidth=2, markersize=8)
ax2.axhline(y=0.95, color='r', linestyle='--', label='95% variance')
ax2.set_xlabel('Principal Component')
ax2.set_ylabel('Cumulative Explained Variance')
ax2.set_title('Cumulative Explained Variance')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Reducing to 2D for visualization

For most datasets, 2 principal components explain a good portion of the variance and allow for easy visualization. Let's apply PCA with just 2 components:

In [ ]:
# Apply PCA with 2 components
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_scaled)

print(f"Shape after PCA: {data_pca.shape}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.4f}")

# Create a DataFrame for easier handling
df_pca = pd.DataFrame(
    data=data_pca,
    columns=[f'PC{i+1}' for i in range(data_pca.shape[1])]
)

df_pca.head()

Now let's visualize the data in PCA space. **Try to see if there are natural groupings in the data, even before we formally cluster:**

In [ ]:
# Plot the PCA result
plt.figure(figsize=(10, 7))

# Color by label if available
if label_column and label_column in df.columns:
    # Get label data aligned with numeric data (after dropna)
    label_data = df[label_column].iloc[df_numeric.index]
    colors = pd.Categorical(label_data).codes
    scatter = plt.scatter(df_pca['PC1'], df_pca['PC2'], 
                          c=colors, cmap='viridis', s=60, alpha=1.0, edgecolors='k', linewidth=0.5)
    plt.colorbar(scatter, label=label_column.capitalize())
else:
    plt.scatter(df_pca['PC1'], df_pca['PC2'], 
               c='steelblue', s=60, alpha=1.0, edgecolors='k', linewidth=0.5)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=11)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=11)
plt.title('Data Projected onto First Two Principal Components', fontsize=12, fontweight='bold')
plt.grid(alpha=0.3)
plt.show()

print(f"Do you see natural groupings in the data?")

## Interpreting PCA components

What do PC1 and PC2 actually represent? We can examine the **loadings** - how much each original variable contributes to each component:

In [ ]:
# Examine loadings
loadings_df = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(pca.n_components_)],
    index=df_numeric.columns
)

print("PCA Loadings (contribution of each variable to each component):")
print(loadings_df.round(3))

# Visualize loadings as a heatmap
plt.figure(figsize=(8, 4))
sns.heatmap(loadings_df, annot=True, fmt='.2f', cmap='RdBu_r', center=0, 
            cbar_kws={'label': 'Loading'})
plt.title('PCA Loadings: How Variables Contribute to Components', fontweight='bold')
plt.ylabel('Original Variables')
plt.xlabel('Principal Components')
plt.tight_layout()
plt.show()

**Understanding the loadings:** 
- High positive or negative values mean that variable strongly influences that component
- Values close to 0 mean the variable has little influence

For the iris dataset, you might notice that PC1 is strongly influenced by petal measurements, while PC2 is influenced by sepal measurements. PCA has automatically found underlying patterns in the data!

# Part 2: Clustering

## What is clustering?

Clustering is a technique for grouping similar observations together *without pre-defined categories*. Unlike classification (where you have labeled examples), clustering discovers groups that emerge from the data itself.

We'll explore three different clustering approaches, each with different strengths:

1. **K-Means**: Fast, interpretable, assumes spherical clusters
2. **Expectation-Maximization (EM/Gaussian Mixture)**: Probabilistic, allows soft assignments
3. **DBSCAN**: Flexible shape, automatically identifies outliers

## K-Means Clustering

**K-Means** works by:
1. Randomly initializing k cluster centers
2. Assigning each point to the nearest center
3. Moving centers to the mean of their assigned points
4. Repeating until centers stop moving

**Key question:** How many clusters (k) should we use? We'll use the **Elbow Method** and **Silhouette Score** to help decide.

### Finding the optimal number of clusters

In [ ]:
# Test different numbers of clusters
inertias = []
silhouette_scores = []
k_values = range(2, 11)

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(data_pca)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(data_pca, kmeans.labels_))

# Plot the Elbow Curve and Silhouette Scores
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(k_values, inertias, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia (within-cluster sum of squares)')
ax1.set_title('Elbow Method: Finding Optimal k')
ax1.grid(alpha=0.3)

ax2.plot(k_values, silhouette_scores, 'go-', linewidth=2, markersize=8)
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score by k')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Silhouette Scores by k:")
for k, score in zip(k_values, silhouette_scores):
    print(f"  k={k}: {score:.4f}")

**Interpreting the plots:**
- The **Elbow Method** looks for an "elbow" where adding more clusters stops helping much
- The **Silhouette Score** (ranges from -1 to 1) measures how well-separated clusters are. Higher is better.

For many datasets, k=3 or k=4 is a good starting point. **Try modifying `n_clusters` below to experiment!**

In [ ]:
# CHOOSE NUMBER OF CLUSTERS HERE
n_clusters = 3
# Try values like 2, 3, 4, 5 to see how clustering changes

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
clusters_kmeans = kmeans.fit_predict(data_pca)

print(f"K-Means with k={n_clusters}")
print(f"Cluster assignments (first 20): {clusters_kmeans[:20]}")
print(f"Cluster sizes: {np.bincount(clusters_kmeans)}")

### Visualizing K-Means results

In [ ]:
# Plot clusters with K-Means
plt.figure(figsize=(10, 7))

# Use Set1 palette for discrete clusters
colors_set1 = plt.cm.Set1(np.linspace(0, 1, n_clusters))

# Plot data points colored by cluster
for cluster_id in range(n_clusters):
    mask = clusters_kmeans == cluster_id
    plt.scatter(df_pca.loc[mask, 'PC1'], df_pca.loc[mask, 'PC2'], 
               c=[colors_set1[cluster_id]], s=60, alpha=1.0, edgecolors='k', 
               linewidth=0.5, label=f'Cluster {cluster_id}')

# Plot cluster centers
centers = kmeans.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], 
           c='black', marker='X', s=400, edgecolors='white', linewidths=2,
           label='Centers', zorder=10)

plt.xlabel('PC1', fontsize=11)
plt.ylabel('PC2', fontsize=11)
plt.title(f'K-Means Clustering (k={n_clusters})', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Expectation-Maximization (EM) / Gaussian Mixture Model

**Expectation-Maximization** is a softer alternative to K-Means. Instead of assigning each point to exactly one cluster, EM estimates the **probability** that each point belongs to each cluster.

**When to use EM:**
- When you have overlapping or fuzzy clusters
- When you want to know confidence levels for cluster membership
- When clusters have different sizes or shapes

In [ ]:
# Apply Gaussian Mixture Model (EM)
em = GaussianMixture(n_components=n_clusters, random_state=42)
clusters_em = em.fit_predict(data_pca)

# Get probability assignments
probabilities = em.predict_proba(data_pca)

print(f"EM with {n_clusters} components")
print(f"Cluster assignments (first 10): {clusters_em[:10]}")
print(f"\nProbabilities for first observation:")
print(f"  {probabilities[0].round(3)}")
print(f"\nCluster sizes: {np.bincount(clusters_em)}")

### Visualizing EM results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Left plot: Clusters
colors_set1 = plt.cm.Set1(np.linspace(0, 1, n_clusters))

for cluster_id in range(n_clusters):
    mask = clusters_em == cluster_id
    ax1.scatter(df_pca.loc[mask, 'PC1'], df_pca.loc[mask, 'PC2'], 
               c=[colors_set1[cluster_id]], s=60, alpha=1.0, edgecolors='k', 
               linewidth=0.5, label=f'Cluster {cluster_id}')

ax1.set_xlabel('PC1', fontsize=11)
ax1.set_ylabel('PC2', fontsize=11)
ax1.set_title('EM Clustering Results', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Right plot: Confidence (posterior probability)
confidence = np.max(probabilities, axis=1)
scatter = ax2.scatter(df_pca['PC1'], df_pca['PC2'], c=confidence, 
                     cmap='viridis', s=60, alpha=1.0, edgecolors='k', linewidth=0.5)
ax2.set_xlabel('PC1', fontsize=11)
ax2.set_ylabel('PC2', fontsize=11)
ax2.set_title('EM Confidence (Posterior Probability)', fontsize=12, fontweight='bold')
cbar = plt.colorbar(scatter, ax=ax2)
cbar.set_label('Max Probability', fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## DBSCAN: Density-Based Clustering

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) finds clusters by looking for areas of high density, separated by areas of low density.

**Key advantages:**
- Automatically discovers number of clusters (no need to specify k)
- Can find clusters of arbitrary shape (not just spheres)
- Identifies outliers as "noise" points (labeled -1)

**Key parameters:**
- `eps`: Maximum distance between points in a cluster
- `min_samples`: Minimum number of points needed to form a dense region

Finding good parameters can take experimentation. Let's try a few values:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# TRY: Adjust eps values to explore different parameter settings
eps_values = [0.1, 0.3, 0.5, 0.7]

for idx, (ax, eps) in enumerate(zip(axes.flat, eps_values)):
    dbscan = DBSCAN(eps=eps, min_samples=5)
    clusters = dbscan.fit_predict(df_pca)
    
    n_clusters_db = len(set(clusters)) - (1 if -1 in clusters else 0)
    n_noise = list(clusters).count(-1)
    
    # Get unique clusters and their colors
    unique_clusters = sorted(set(clusters))
    
    # Use Set1 for regular clusters, gray for noise
    colors_set1 = plt.cm.Set1(np.linspace(0, 1, max(n_clusters_db, 1)))
    
    for cluster_id in unique_clusters:
        if cluster_id == -1:
            # Noise points in gray
            mask = clusters == -1
            ax.scatter(df_pca.loc[mask, 'PC1'], df_pca.loc[mask, 'PC2'],
                      c='gray', s=60, alpha=1.0, marker='x', linewidth=1,
                      label='Noise')
        else:
            mask = clusters == cluster_id
            ax.scatter(df_pca.loc[mask, 'PC1'], df_pca.loc[mask, 'PC2'],
                      c=[colors_set1[cluster_id]], s=60, alpha=1.0, edgecolors='k',
                      linewidth=0.5, label=f'Cluster {cluster_id}')
    
    ax.set_xlabel('PC1', fontsize=10)
    ax.set_ylabel('PC2', fontsize=10)
    ax.set_title(f'eps={eps} (Clusters: {n_clusters_db}, Noise: {n_noise})', 
                fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()



**Choose a good eps value** based on the results above. Points marked with red X are noise (outliers). **Try modifying `eps` below to experiment:**

In [ ]:
# CHOOSE DBSCAN PARAMETERS HERE
eps = 0.5
min_samples = 5

dbscan = DBSCAN(eps=eps, min_samples=min_samples)
clusters_dbscan = dbscan.fit_predict(data_pca)

n_clusters_found = len(set(clusters_dbscan)) - (1 if -1 in clusters_dbscan else 0)
n_noise = list(clusters_dbscan).count(-1)

print(f"DBSCAN Results (eps={eps}, min_samples={min_samples}):")
print(f"  Number of clusters: {n_clusters_found}")
print(f"  Number of noise points: {n_noise}")
print(f"  Cluster sizes: {np.bincount(clusters_dbscan[clusters_dbscan >= 0])}")

In [ ]:
plt.figure(figsize=(10, 7))

# Get DBSCAN results with optimal parameters
dbscan = DBSCAN(eps=0.5, min_samples=5)
clusters_dbscan = dbscan.fit_predict(df_pca)

n_clusters_db = len(set(clusters_dbscan)) - (1 if -1 in clusters_dbscan else 0)
n_noise = list(clusters_dbscan).count(-1)

# Use Set1 for regular clusters
colors_set1 = plt.cm.Set1(np.linspace(0, 1, max(n_clusters_db, 1)))
unique_clusters = sorted(set(clusters_dbscan))

for cluster_id in unique_clusters:
    if cluster_id == -1:
        # Noise points in gray
        mask = clusters_dbscan == -1
        plt.scatter(df_pca.loc[mask, 'PC1'], df_pca.loc[mask, 'PC2'],
                   c='gray', s=60, alpha=1.0, marker='x', linewidth=1.5,
                   label='Noise', zorder=5)
    else:
        mask = clusters_dbscan == cluster_id
        plt.scatter(df_pca.loc[mask, 'PC1'], df_pca.loc[mask, 'PC2'],
                   c=[colors_set1[cluster_id]], s=60, alpha=1.0, edgecolors='k',
                   linewidth=0.5, label=f'Cluster {cluster_id}')

plt.xlabel('PC1', fontsize=11)
plt.ylabel('PC2', fontsize=11)
plt.title(f'DBSCAN Clustering (eps=0.5, min_samples=5)\nClusters: {n_clusters_db}, Noise Points: {n_noise}', 
         fontsize=12, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Comparing clustering methods

Let's compare all three clustering methods side-by-side:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Prepare cluster results for each method
methods_results = [
    ('K-Means', clusters_kmeans, n_clusters),
    ('EM (GMM)', clusters_em, n_clusters),
    ('DBSCAN', clusters_dbscan, n_clusters_db if n_clusters_db > 0 else 1)
]

for ax, (method_name, clusters, max_cluster_id) in zip(axes, methods_results):
    colors_set1 = plt.cm.Set1(np.linspace(0, 1, max(max_cluster_id, 1)))
    unique_clusters = sorted(set(clusters))
    
    for cluster_id in unique_clusters:
        if cluster_id == -1:
            # DBSCAN noise points
            mask = clusters == -1
            ax.scatter(df_pca.loc[mask, 'PC1'], df_pca.loc[mask, 'PC2'],
                      c='gray', s=60, alpha=1.0, marker='x', linewidth=1.5,
                      label='Noise')
        else:
            mask = clusters == cluster_id
            ax.scatter(df_pca.loc[mask, 'PC1'], df_pca.loc[mask, 'PC2'],
                      c=[colors_set1[cluster_id]], s=60, alpha=1.0, edgecolors='k',
                      linewidth=0.5, label=f'Cluster {cluster_id}')
    
    ax.set_xlabel('PC1', fontsize=10)
    ax.set_ylabel('PC2', fontsize=10)
    ax.set_title(method_name, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9, loc='best')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Clustering Comparison ===")
print(f"K-Means: {n_clusters} clusters")
print(f"EM (GMM): {n_clusters} clusters")
print(f"DBSCAN: {n_clusters_db} clusters, {n_noise} noise points")

## When to use each method

| Method | Best for | Strengths | Limitations |
|--------|----------|-----------|-------------|
| **K-Means** | General clustering, spherical clusters | Fast, simple, interpretable | Must specify k, assumes equal-sized clusters |
| **EM** | Overlapping clusters, probabilistic assignment | Handles soft membership, flexible | Computationally intensive, must specify k |
| **DBSCAN** | Irregular shapes, outlier detection | Auto-finds clusters, handles noise | Sensitive to parameters, struggles with varying densities |

**For this dataset, which method seems most effective?** Look at the visualizations and think about whether clusters are well-separated, overlapping, or have irregular shapes.

## Summary

In this notebook, you've learned:

1. **Dimensionality Reduction (PCA)**: How to transform high-dimensional data into a lower-dimensional space that preserves the most important variation
2. **Cluster Evaluation**: How to determine the right number of clusters using the elbow method and silhouette scores
3. **K-Means**: A fast, interpretable clustering algorithm for spherical clusters
4. **Expectation-Maximization**: A probabilistic approach that handles overlapping clusters and uncertainty
5. **DBSCAN**: A density-based method that finds arbitrary-shaped clusters and identifies outliers
6. **Practical workflow**: Load data → Standardize → Reduce dimensions → Cluster → Visualize → Iterate

These techniques are foundational in data science and used across many domains - from customer segmentation to image analysis to geographic clustering. The key to mastery is experimentation: try different datasets, different numbers of clusters, and different parameter values to build intuition about how these methods behave.

## Resources for further learning

- [scikit-learn: Clustering](https://scikit-learn.org/stable/modules/clustering.html)
- [scikit-learn: Decomposition and Dimensionality Reduction](https://scikit-learn.org/stable/modules/decomposition.html)
- [Understanding PCA](https://towardsdatascience.com/a-one-stop-shop-for-principal-component-analysis-5582fb7e5b9c)
- [K-Means Deep Dive](https://towardsdatascience.com/understanding-k-means-clustering-in-machine-learning-6a6e3a08cd2f)
- [DBSCAN Explained](https://towardsdatascience.com/dbscan-clustering-for-data-shapes-k-means-cant-handle-well-d46b76331031)